# Give Me Some Credit -- Cleaning, EDA, Feature Engineering & Split

Merged notebook: combines `give_me_some_credit.ipynb` (step-by-step cleaning) and `Main_EDA.ipynb` (exploratory analysis), then adds feature engineering (per the FE spec) and a stratified train/test split.

1. Setup
2. Part 1 -- Data Cleaning (step-by-step)
3. Part 2 -- Exploratory Data Analysis
4. Part 3 -- Feature Engineering (`build_gmsc_features`, self-contained -- reads and cleans the raw CSV per the FE spec independently of Part 1)
5. Part 4 -- Train / test split
6. Part 5 -- Save + download

## 0) Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

sns.set_theme(style="whitegrid", font_scale=0.95)
PALETTE = ["#2E4057", "#D1495B", "#66A182", "#EDAE49", "#8367C7"]
sns.set_palette(PALETTE)

RANDOM_STATE = 42

# single source of truth for the file path -- used by every section below
GMSC_PATH = "/content/give me some credit.csv"


## Part 1 -- Data Cleaning (step-by-step)

Reloads the raw file fresh into its own `df` (independent of Part 1's EDA `df`, which picked up extra EDA-only helper columns) and applies the full cleaning pipeline: column-name standardization, whitespace/NaN normalization, dtype fixes, de-duplication, impossible-value handling, empty/constant-column removal, and a leakage check.

In [ ]:
# 1) Load the data
df = pd.read_csv(GMSC_PATH)

print("Shape before cleaning:", df.shape)
df.info()


In [ ]:
# 2) Standardize column names

df.rename(columns={"Unnamed: 0": "ID"}, inplace=True)

df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace(r"[^A-Za-z0-9_]", "", regex=True)
)

print("\nColumn names after standardization:")
print(df.columns.tolist())


In [ ]:
# 3) Strip extra whitespace from text values

obj_cols = df.select_dtypes(include="object").columns
for col in obj_cols:
    df[col] = df[col].astype(str).str.strip()

In [ ]:
# 4) Convert "empty-like" values into actual NaN

na_like = ["NA", "N/A", "na", "n/a", "", "null", "NULL", "None", "-", "?"]
df.replace(na_like, np.nan, inplace=True)

print("\nMissing value counts per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# 5) Fix data types

for col in df.select_dtypes(include="object").columns:
    converted = pd.to_numeric(df[col], errors="coerce")
    if converted.notna().sum() >= 0.9 * df[col].notna().sum():
        df[col] = converted

print("\nDtypes after fixing:")
print(df.dtypes)


In [ ]:
# 6) Remove true duplicates

before = df.shape[0]
df.drop_duplicates(inplace=True)

if "ID" in df.columns:
    dup_ids = df["ID"].duplicated().sum()
    print(f"\nNumber of duplicated IDs: {dup_ids}")
    df.drop_duplicates(subset="ID", keep="first", inplace=True)

print(f"Removed {before - df.shape[0]} duplicate rows")

In [ ]:
# 7) Detect impossible values

# age must be within a realistic human range
invalid_age = df[(df["age"] < 18) | (df["age"] > 100)]
print(f"\nNumber of invalid age rows: {len(invalid_age)}")
df.loc[(df["age"] < 18) | (df["age"] > 100), "age"] = np.nan

# NumberOfTime30-59 / 60-89 / 90DaysLate columns: 96 and 98 are known
# data-entry error codes in this dataset, not real counts -> treat as NaN
late_cols = [
    "NumberOfTime3059DaysPastDueNotWorse",
    "NumberOfTime6089DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]
late_cols = [c for c in late_cols if c in df.columns]
for col in late_cols:
    bad = df[col].isin([96, 98]).sum()
    print(f"Number of error-code values (96/98) in {col}: {bad}")
    df.loc[df[col].isin([96, 98]), col] = np.nan

# RevolvingUtilizationOfUnsecuredLines should logically be a ratio (~0 to a bit above 1)
# extreme values (e.g. in the thousands) are data errors
if "RevolvingUtilizationOfUnsecuredLines" in df.columns:
    extreme = (df["RevolvingUtilizationOfUnsecuredLines"] > 10).sum()
    print(f"Extreme RevolvingUtilizationOfUnsecuredLines values (>10): {extreme}")
    df.loc[df["RevolvingUtilizationOfUnsecuredLines"] > 10, "RevolvingUtilizationOfUnsecuredLines"] = np.nan

# DebtRatio has extreme outliers that are almost certainly data errors
if "DebtRatio" in df.columns:
    extreme_dr = (df["DebtRatio"] > 10).sum()
    print(f"Extreme DebtRatio values (>10): {extreme_dr}")
    # flagged here, not auto-removed - decide later during outlier treatment
    df["DebtRatio_flag_extreme"] = df["DebtRatio"] > 10

# MonthlyIncome and NumberOfDependents should not be negative
for col in ["MonthlyIncome", "NumberOfDependents"]:
    if col in df.columns:
        neg = (df[col] < 0).sum()
        print(f"Negative values in {col}: {neg}")
        df.loc[df[col] < 0, col] = np.nan

In [ ]:
# 8) Standardize categorical values
# SeriousDlqin2yrs is already a clean binary target (0/1) - nothing to unify
print("\nSeriousDlqin2yrs distribution:")
print(df["SeriousDlqin2yrs"].value_counts())

In [ ]:
# 9) Drop columns that are fully empty or constant

empty_cols = [c for c in df.columns if df[c].isnull().all()]
constant_cols = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]

cols_to_drop = list(set(empty_cols + constant_cols))
print(f"\nFully empty columns: {empty_cols}")
print(f"Constant columns (only one unique value): {constant_cols}")

df.drop(columns=cols_to_drop, inplace=True)

In [ ]:
# 10) Check for leakage columns

target_col = "SeriousDlqin2yrs"
print(f"\nTarget column: {target_col}")

id_like_cols = [c for c in df.columns if c.upper() == "ID" or c.upper().endswith("_ID")]
print(f"ID-like columns (should be excluded from the model): {id_like_cols}")

numeric_df = df.select_dtypes(include=[np.number]).drop(columns=id_like_cols, errors="ignore")
if target_col in numeric_df.columns:
    corr = numeric_df.corr()[target_col].sort_values(ascending=False)
    print("\nCorrelation with target:")
    print(corr)
    suspicious = corr[(corr.abs() > 0.9) & (corr.index != target_col)]
    print(f"\nSuspicious potential leakage columns (correlation > 0.9): {suspicious.index.tolist()}")


In [ ]:
# 11) Final look at the cleaned data
print("\nShape after cleaning:", df.shape)
print("\nSummary statistics:")
print(df.describe(include="all").T)

## Part 2 -- Exploratory Data Analysis (raw data)

## 1. Load the data & first look

In [ ]:
df_raw = pd.read_csv(GMSC_PATH)
df = df_raw.rename(columns={"Unnamed: 0": "ID"}).copy()

print("Shape:", df.shape)
df.info()


In [ ]:
df.head()

In [ ]:
df.describe(include="all").T

## 2. Data quality audit

We check, in order: missing values, exact/near duplicates, and known
"impossible" or sentinel values that are common in this specific dataset
(e.g. age = 0, and the 96 / 98 special codes in the past-due counters).


### 2.1 Missing values

In [ ]:
na_counts = df.isnull().sum()
na_pct = (na_counts / len(df) * 100).round(2)
na_table = pd.DataFrame({"missing_count": na_counts, "missing_%": na_pct})
na_table = na_table[na_table["missing_count"] > 0].sort_values("missing_%", ascending=False)
na_table


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
if len(na_table):
    ax.barh(na_table.index, na_table["missing_%"], color=PALETTE[1])
    for i, v in enumerate(na_table["missing_%"]):
        ax.text(v + 0.3, i, f"{v}%", va="center", fontsize=9)
    ax.set_xlabel("% missing")
    ax.set_title("Missing values by column")
else:
    ax.text(0.5, 0.5, "No missing values", ha="center")
plt.tight_layout()
plt.show()


**Finding:** two columns have missing data — `MonthlyIncome` (~19.8%) and
`NumberOfDependents` (~2.6%). Both are plausible "not reported" fields rather
than random glitches; we test whether they're missing-at-random in Section 8.

### 2.2 Duplicates

In [ ]:
exact_dupes = df.duplicated().sum()
dupes_ignoring_id = df.drop(columns=["ID"]).duplicated().sum()
print(f"Fully identical rows (incl. ID): {exact_dupes}")
print(f"Identical on every column EXCEPT the ID index: {dupes_ignoring_id}")


**Finding:** there are no duplicate `ID`s (it's just a row index), but
**609 rows are duplicated on every real feature** once `ID` is excluded.
These are worth flagging for modeling (they can leak between train/test splits)
even though we won't drop them in this EDA pass.

### 2.3 Impossible / sentinel values

In [ ]:
print("age == 0 rows:", (df["age"] == 0).sum())
print("age range:", df["age"].min(), "-", df["age"].max())
df[df["age"] == 0]


In [ ]:
late_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]

sentinel_mask = (df[late_cols] >= 96).any(axis=1)
print(f"Rows containing a 96/98 sentinel code in any 'DaysPastDue' column: {sentinel_mask.sum()}")

for c in late_cols:
    vc = df[c].value_counts().sort_index(ascending=False).head(6)
    print(f"\nTop values in {c}:")
    print(vc)


In [ ]:
print("Default rate for sentinel rows (>=96):", df.loc[sentinel_mask, "SeriousDlqin2yrs"].mean().round(4))
print("Default rate overall                    :", df["SeriousDlqin2yrs"].mean().round(4))
print("MonthlyIncome missing rate in sentinel rows:", df.loc[sentinel_mask, "MonthlyIncome"].isnull().mean().round(4))


**Finding — important data quality issue:** 269 rows (≈0.18%) carry the values
**96 or 98** in all three "days past due" columns simultaneously. These are not
real counts of 96–98 late payments; they look like **coded/sentinel values**
(a common artifact of this specific Kaggle dataset). This small subgroup has a
**default rate of ~55% vs. ~6.7% overall** — an 8x lift — and much higher
missingness in `MonthlyIncome`. This group should be handled explicitly
(flag as a separate category or cap/clip) rather than treated as ordinary
large counts, or it will distort any distance/linear-based model.

In [ ]:
print("RevolvingUtilizationOfUnsecuredLines > 1  :", (df["RevolvingUtilizationOfUnsecuredLines"] > 1).sum())
print("RevolvingUtilizationOfUnsecuredLines > 10 :", (df["RevolvingUtilizationOfUnsecuredLines"] > 10).sum())
print("max value                                  :", df["RevolvingUtilizationOfUnsecuredLines"].max())
print()
print("DebtRatio > 1  :", (df["DebtRatio"] > 1).sum())
print("DebtRatio > 10 :", (df["DebtRatio"] > 10).sum())
print("max value      :", df["DebtRatio"].max())


**Finding:** `RevolvingUtilizationOfUnsecuredLines` is conceptually a ratio
that should sit in **[0, 1]**, yet it reaches **50,708** and 3,321 rows exceed 1.
`DebtRatio` shows the same pattern (max ≈ 329,664; 35,137 rows > 1). These are
almost certainly data-entry errors or unit mismatches, not genuine values, and
they will dominate any mean/std-based scaling unless winsorized, log-transformed,
or capped.

## 3. Target variable — class balance

In [ ]:
target = "SeriousDlqin2yrs"
vc = df[target].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(5, 3.6))
bars = ax.bar(["0 · no serious delinquency", "1 · serious delinquency"], vc.values,
              color=[PALETTE[0], PALETTE[1]])
for b, v in zip(bars, vc.values):
    ax.text(b.get_x() + b.get_width()/2, v, f"{v:,}\n({v/len(df):.2%})",
            ha="center", va="bottom")
ax.set_ylim(0, vc.max()*1.2)
ax.set_title("Target distribution")
plt.tight_layout()
plt.show()

print(f"Positive (default) rate: {df[target].mean():.2%}")
print(f"Imbalance ratio: {(1-df[target].mean())/df[target].mean():.1f} : 1")


**Finding:** the dataset is **strongly imbalanced** — only **6.68%** of
borrowers defaulted. Accuracy alone will be a misleading metric; ROC-AUC /
PR-AUC, class weighting, or resampling should be used downstream.

## 4. Univariate analysis

Distribution of every feature, using a log-scale where the raw scale is too
skewed to read (utilization, debt ratio, income).

In [ ]:
feature_cols = [c for c in df.columns if c not in ["ID", target]]
df[feature_cols].describe().T


In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 14))
axes = axes.ravel()

plot_specs = {
    "RevolvingUtilizationOfUnsecuredLines": dict(clip=(0, 2), bins=60),
    "age": dict(clip=None, bins=50),
    "NumberOfTime30-59DaysPastDueNotWorse": dict(clip=(0, 10), bins=11),
    "DebtRatio": dict(clip=(0, 2), bins=60),
    "MonthlyIncome": dict(clip=(0, 20000), bins=60),
    "NumberOfOpenCreditLinesAndLoans": dict(clip=None, bins=40),
    "NumberOfTimes90DaysLate": dict(clip=(0, 10), bins=11),
    "NumberRealEstateLoansOrLines": dict(clip=(0, 10), bins=11),
    "NumberOfTime60-89DaysPastDueNotWorse": dict(clip=(0, 10), bins=11),
    "NumberOfDependents": dict(clip=None, bins=12),
}

for ax, col in zip(axes, feature_cols):
    spec = plot_specs.get(col, dict(clip=None, bins=40))
    data = df[col].dropna()
    if spec["clip"] is not None:
        lo, hi = spec["clip"]
        data = data[(data >= lo) & (data <= hi)]
        suffix = f" (clipped to [{lo},{hi}] for readability)"
    else:
        suffix = ""
    ax.hist(data, bins=spec["bins"], color=PALETTE[0], edgecolor="white")
    ax.set_title(col + suffix, fontsize=9)
    ax.tick_params(labelsize=8)

for ax in axes[len(feature_cols):]:
    ax.axis("off")

plt.tight_layout()
plt.show()


**Findings:**
- `age` is roughly bell-shaped, centered in the 40s-50s, with one impossible `age=0` row.
- `RevolvingUtilizationOfUnsecuredLines` and `DebtRatio` are extremely right-skewed with a long tail of implausible outliers (see §2.3).
- The three "days past due" counters are heavily zero-inflated (most borrowers have 0 late payments) with a small sentinel spike at 96/98.
- `MonthlyIncome` is right-skewed, typical of income data — a log transform will help most models.
- `NumberOfDependents` and `NumberRealEstateLoansOrLines` are low-count discrete variables, also zero-inflated.

## 5. Outlier detection (boxplots + IQR rule)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
axes = axes.ravel()
box_cols = ["RevolvingUtilizationOfUnsecuredLines", "age", "DebtRatio",
            "MonthlyIncome", "NumberOfOpenCreditLinesAndLoans", "NumberOfTimes90DaysLate"]

for ax, col in zip(axes, box_cols):
    sns.boxplot(y=df[col].dropna(), ax=ax, color=PALETTE[2])
    ax.set_title(col, fontsize=9)
    ax.set_ylabel("")

plt.tight_layout()
plt.show()


In [ ]:
def iqr_outlier_report(series, name):
    s = series.dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = ((s < lo) | (s > hi)).sum()
    return {"feature": name, "q1": q1, "q3": q3, "IQR_low": lo, "IQR_high": hi,
            "n_outliers": n_out, "pct_outliers": round(n_out/len(s)*100, 2)}

report = pd.DataFrame([iqr_outlier_report(df[c], c) for c in feature_cols]).set_index("feature")
report.sort_values("pct_outliers", ascending=False)


**Finding:** by the standard 1.5×IQR rule, `RevolvingUtilizationOfUnsecuredLines`,
`DebtRatio`, and the past-due counters show the highest outlier rates — consistent
with the sentinel-code and unit-error issues already identified above rather than
genuine extreme-but-valid customers.

## 6. Bivariate analysis — features vs. target

For each feature we compare the distribution/mean between defaulters (1) and
non-defaulters (0), and show the default rate across bins.

In [ ]:
def default_rate_by_bin(data, col, target, bins=10, clip=None):
    d = data[[col, target]].dropna().copy()
    if clip is not None:
        d = d[(d[col] >= clip[0]) & (d[col] <= clip[1])]
    d["bin"] = pd.qcut(d[col], q=bins, duplicates="drop")
    out = d.groupby("bin", observed=True)[target].agg(["mean", "count"])
    out.columns = ["default_rate", "n"]
    return out

fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.ravel()

bivar_cols = ["RevolvingUtilizationOfUnsecuredLines", "age", "DebtRatio",
              "MonthlyIncome", "NumberOfOpenCreditLinesAndLoans",
              "NumberOfTime30-59DaysPastDueNotWorse", "NumberOfTimes90DaysLate",
              "NumberRealEstateLoansOrLines", "NumberOfDependents"]

for ax, col in zip(axes, bivar_cols):
    try:
        tab = default_rate_by_bin(df, col, target, bins=8)
        ax.plot(range(len(tab)), tab["default_rate"], marker="o", color=PALETTE[1])
        ax.set_xticks(range(len(tab)))
        ax.set_xticklabels([str(i) for i in tab.index], rotation=45, ha="right", fontsize=6)
    except Exception as e:
        ax.text(0.5, 0.5, f"n/a: {e}", ha="center", fontsize=7)
    ax.axhline(df[target].mean(), color="gray", linestyle="--", linewidth=1)
    ax.set_title(f"Default rate by {col} decile", fontsize=8)
    ax.set_ylabel("default rate")

plt.tight_layout()
plt.show()


**Findings (dashed line = overall base rate of 6.68%):**
- Default rate **rises sharply with age going down** — younger borrowers default noticeably more; it falls steadily into older age brackets.
- Default rate **increases monotonically** with the past-due counters and with `RevolvingUtilizationOfUnsecuredLines` — both behave exactly as expected of strong risk signals.
- `MonthlyIncome` shows a mild inverse relationship — lower income bands skew toward higher default.
- `NumberOfOpenCreditLinesAndLoans` and `NumberRealEstateLoansOrLines` show a much flatter/non-monotonic relationship — weaker standalone signal, may still interact with other features.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 11))
axes = axes.ravel()

for ax, col in zip(axes, bivar_cols):
    d = df[[col, target]].dropna()
    lo, hi = d[col].quantile([0.01, 0.99])
    d = d[(d[col] >= lo) & (d[col] <= hi)]
    sns.boxplot(data=d, x=target, y=col, ax=ax, palette=[PALETTE[0], PALETTE[1]])
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("SeriousDlqin2yrs")

plt.tight_layout()
plt.show()


## 7. Correlation & multivariate structure

In [ ]:
corr = df[feature_cols + [target]].corr(method="spearman")

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax,
            cbar_kws={"label": "Spearman correlation"})
ax.set_title("Spearman correlation matrix (robust to outliers/skew)")
plt.tight_layout()
plt.show()


In [ ]:
corr_with_target = corr[target].drop(target).sort_values(key=abs, ascending=False)
corr_with_target.to_frame("spearman_corr_with_target")


**Findings:**
- The three "days past due" counters are **highly correlated with each other**
  (they measure the same underlying behavior at different severities) and are
  the strongest correlates of the target — expected, since they are the most
  direct historical proxies for delinquency.
- `RevolvingUtilizationOfUnsecuredLines` is the single strongest standalone
  correlate with the target.
- `age` correlates negatively with the target (older ⇒ lower risk), and mildly
  negatively with the past-due counters too.
- `DebtRatio`, `MonthlyIncome`, `NumberOfOpenCreditLinesAndLoans`, and
  `NumberRealEstateLoansOrLines` show comparatively weak linear/rank correlation
  with the target on their own — they will likely help mainly through interactions
  or non-linear models rather than as strong solo predictors.

## 8. Missingness patterns — is data missing at random?

We test whether `MonthlyIncome` and `NumberOfDependents` are missing completely
at random, or whether missingness itself correlates with default risk (MNAR/MAR),
which matters for the correct imputation strategy.

In [ ]:
df["MonthlyIncome_missing"] = df["MonthlyIncome"].isnull().astype(int)
df["NumberOfDependents_missing"] = df["NumberOfDependents"].isnull().astype(int)

for flag in ["MonthlyIncome_missing", "NumberOfDependents_missing"]:
    rate_missing = df.loc[df[flag] == 1, target].mean()
    rate_present = df.loc[df[flag] == 0, target].mean()
    print(f"{flag}: default rate when MISSING = {rate_missing:.2%}, "
          f"when PRESENT = {rate_present:.2%}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, flag in zip(axes, ["MonthlyIncome_missing", "NumberOfDependents_missing"]):
    rates = df.groupby(flag)[target].mean()
    ax.bar(["present", "missing"], rates.values, color=[PALETTE[0], PALETTE[1]])
    ax.axhline(df[target].mean(), color="gray", linestyle="--", linewidth=1)
    ax.set_title(flag)
    ax.set_ylabel("default rate")
plt.tight_layout()
plt.show()


**Finding:** default rate is **meaningfully higher when `MonthlyIncome` is
missing** than when it's reported — this points to **MAR/MNAR rather than
MCAR**, i.e. missingness itself carries signal. A missing-value indicator
feature (kept alongside imputed values) is recommended rather than silent
mean/median imputation.

## 9. Summary of key findings & recommendations

**Data quality issues found**
1. One row has `age = 0` (impossible) — should be treated as missing/dropped.
2. 269 rows carry sentinel codes **96/98** across all three "days past due"
   columns — a distinct high-risk subgroup (≈55% default rate) that should be
   flagged rather than treated as literal counts.
3. `RevolvingUtilizationOfUnsecuredLines` and `DebtRatio` contain extreme,
   almost certainly erroneous outliers (max values in the tens of thousands
   for what should be bounded ratios) — cap/winsorize or log-transform.
4. 609 rows are exact duplicates on every real feature (different `ID` only) —
   worth deduplicating before train/test splitting to avoid leakage.
5. `MonthlyIncome` (~19.8%) and `NumberOfDependents` (~2.6%) have missing
   values, and missingness is **not random** — it correlates with higher
   default risk, so an explicit "is-missing" indicator should be kept.

**Target & modeling implications**
- Severe class imbalance (6.68% positive) → use ROC-AUC/PR-AUC, class weights,
  or resampling; avoid plain accuracy.
- Strongest predictors: the past-due counters and revolving utilization;
  age is protective; income, debt ratio, and credit-line counts are weaker
  standalone signals but may add value through interactions or tree-based
  models that handle non-linearities and outliers more gracefully than
  linear models.
- Recommended preprocessing before modeling: cap/winsorize utilization and
  debt ratio, isolate or flag the 96/98 sentinel rows, impute income/dependents
  with a missing-indicator retained, and consider log-transforming income.


## Part 3 -- Feature Engineering

`build_gmsc_features` is self-contained: it reads `GMSC_PATH` and performs its own cleaning steps (income-missing flag, debt-ratio fix, 96/98 placeholder fix, age fix, duplicate flag, dependents fill) that mirror Part 2 but follow the FE spec's exact rules, plus every derived feature required for modeling. This is independent of the manually-cleaned `df` from Part 2 (which stays available above for inspection).

In [ ]:
def build_gmsc_features(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    # -- duplicate review: flag rows identical except the row-id column ----
    id_like_cols = [c for c in df.columns if c.startswith("Unnamed")]
    dup_check_cols = [c for c in df.columns if c not in id_like_cols]
    duplicate_flag = df.duplicated(subset=dup_check_cols, keep=False).astype(int)
    df = df.drop(columns=id_like_cols)

    out = pd.DataFrame(index=df.index)
    out["duplicate_flag"] = duplicate_flag.values

    # -- income_missing_flag (before any use of income) --------------------
    out["income_missing_flag"] = df["MonthlyIncome"].isna().astype(int)

    # income: REAL
    out["income"] = df["MonthlyIncome"]

    # -- debt_ratio fix: NaN when income missing/0; flag outlier > 5 -------
    valid_income_mask = out["income"].notna() & (out["income"] > 0)
    debt_ratio_clean = df["DebtRatio"].where(valid_income_mask)
    out["derived_debt_ratio_outlier"] = (df["DebtRatio"] > 5).astype(int)

    # -- placeholder code fix: 96/98 in past-due cols -> NaN + flag --------
    past_due_cols = [
        "NumberOfTime30-59DaysPastDueNotWorse",
        "NumberOfTime60-89DaysPastDueNotWorse",
        "NumberOfTimes90DaysLate",
    ]
    placeholder_mask = pd.Series(False, index=df.index)
    past_due_clean = pd.DataFrame(index=df.index)
    for col in past_due_cols:
        is_placeholder = df[col].isin([96, 98])
        placeholder_mask = placeholder_mask | is_placeholder
        past_due_clean[col] = df[col].where(~is_placeholder)
    out["derived_placeholder_code_flag"] = placeholder_mask.astype(int)

    # -- age fix: 0 -> NaN ---------------------------------------------------
    out["age"] = df["age"].replace(0, np.nan)

    # -- dependents_missing_flag + fill(0) ----------------------------------
    out["dependents_missing_flag"] = df["NumberOfDependents"].isna().astype(int)
    out["dependents"] = df["NumberOfDependents"].fillna(0)

    # -- derived_late_count / derived_late_severity (post placeholder-fix) --
    out["derived_late_count"] = past_due_clean.sum(axis=1, skipna=True)
    out["derived_late_severity"] = (
        past_due_clean["NumberOfTime30-59DaysPastDueNotWorse"].fillna(0) * 1
        + past_due_clean["NumberOfTime60-89DaysPastDueNotWorse"].fillna(0) * 2
        + past_due_clean["NumberOfTimes90DaysLate"].fillna(0) * 3
    )
    # late_payments: harmonized column = the cleaned late count
    out["late_payments"] = out["derived_late_count"]

    # -- derived_obligations: debt_ratio * income, only when income>0 AND
    # debt_ratio<=5 (per FE.pdf rule) --------------------------------------
    obligations_mask = valid_income_mask & (df["DebtRatio"] <= 5)
    out["derived_obligations"] = np.where(
        obligations_mask, df["DebtRatio"] * out["income"], np.nan
    )
    # monthly_debt_payment: PROXY = derived_obligations (harmonized name)
    out["monthly_debt_payment"] = out["derived_obligations"]
    out["monthly_debt_payment_is_proxy"] = 1

    # derived_expense_ratio = cleaned debt ratio
    out["derived_expense_ratio"] = debt_ratio_clean

    # derived_monthly_surplus = income - obligations
    out["derived_monthly_surplus"] = out["income"] - out["derived_obligations"]
    out["savings"] = out["derived_monthly_surplus"]
    out["savings_is_proxy"] = 1

    # derived_overspend_flag = debt_ratio > 1 (cleaned)
    out["derived_overspend_flag"] = (debt_ratio_clean > 1).astype(int)

    # derived_utilization_capped = min(utilization, 1) + outlier flag
    util = df["RevolvingUtilizationOfUnsecuredLines"]
    out["derived_utilization_capped"] = util.clip(upper=1)
    out["derived_utilization_outlier_flag"] = (util > 1).astype(int)
    out["revolving_utilization_ref"] = util

    # derived_log_income = log1p(income)
    out["derived_log_income"] = np.log1p(out["income"].clip(lower=0))

    # debt: MISSING (no outstanding-balance dollar figure exists in GMSC)
    out["debt"] = np.nan

    # expenses: MISSING (no separate living-expense figure in GMSC)
    out["expenses"] = np.nan

    # rent: MISSING
    out["rent"] = np.nan

    # income_stability: MISSING (single snapshot, no history)
    out["income_stability"] = np.nan

    # month: MISSING (no time dimension in GMSC)
    out["month"] = np.nan

    out["target"] = df["SeriousDlqin2yrs"]
    out["source_dataset"] = "gmsc"
    return out


In [ ]:
gmsc = build_gmsc_features(GMSC_PATH)
print("GMSC feature shape:", gmsc.shape)
gmsc.head()


## Part 4 -- Train / test split

In [ ]:
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.2
RANDOM_STATE = 42

# GMSC: one row per client -> plain stratified split on the target
gmsc_train, gmsc_test = train_test_split(
    gmsc,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=gmsc["target"],
)

print("GMSC train:", gmsc_train.shape, " test:", gmsc_test.shape)
print("GMSC train target rate:", round(gmsc_train["target"].mean(), 4),
      "| test target rate:", round(gmsc_test["target"].mean(), 4))


## Part 5 -- Save + download

In [ ]:
gmsc.to_csv("gmsc_target_features.csv", index=False)
gmsc_train.to_csv("gmsc_train.csv", index=False)
gmsc_test.to_csv("gmsc_test.csv", index=False)

_OUT_FILES = ["gmsc_target_features.csv", "gmsc_train.csv", "gmsc_test.csv"]

try:
    from google.colab import files as _files
    for _f in _OUT_FILES:
        _files.download(_f)
except ImportError:
    print("Saved locally:", ", ".join(_OUT_FILES))
